In [1]:
%%capture
! pip install langchain
! pip install -qU "langchain[openai]"
! pip install -qU langchain_community beautifulsoup4

In [2]:
import re
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import CharacterTextSplitter

In [3]:
def pre_processing(text: str) -> str:
    text = re.sub(r'\n+', '\n', text)
    text = re.sub(r'\ +', ' ', text)
    return text


In [4]:
urls = ["https://www.orange.ci/fr/max-it/super-app-orange-et-moi-orange-money.html",
        "https://www.orange.ci/fr/comment-obtenir-sa-facture-fixe-mobile-ou-internet-orange.html",
        "https://www.orange.ci/fr/comment-debloquer-mon-compte-orange-money.html"]
loader = WebBaseLoader(urls)
docs_loader = loader.load()

In [5]:
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", chunk_size=100, chunk_overlap=10, separator="\n"
)

In [6]:
docs = text_splitter.split_documents(docs_loader)

In [7]:
for i, doc in enumerate(docs):
    doc.metadata["split_number"] = i + 1
    doc.page_content=pre_processing(doc.page_content)

In [8]:
(docs)

[Document(metadata={'source': 'https://www.orange.ci/fr/max-it/super-app-orange-et-moi-orange-money.html', 'title': 'Max it, la super app qui fusionne Orange et moi et Orange Money ! | Orange Côte d’Ivoire', 'description': 'Nouveau, la super app d’Orange vient de sortir ! C’est Max it, la fusion des applis Orange et moi avec Orange Money ! Achats de pass, transferts d’argent et paiement marchand avec Qr code, achats de tickets de concerts dans Marketplace, et plus encore…', 'language': 'fr', 'split_number': 1}, page_content='Max it, la super app qui fusionne Orange et moi et Orange Money ! | Orange Côte d’Ivoire\nQuand les applis Orange et moi et Orange Money fusionnent, ça donne la super app Max it !'),
 Document(metadata={'source': 'https://www.orange.ci/fr/max-it/super-app-orange-et-moi-orange-money.html', 'title': 'Max it, la super app qui fusionne Orange et moi et Orange Money ! | Orange Côte d’Ivoire', 'description': 'Nouveau, la super app d’Orange vient de sortir ! C’est Max it,

In [9]:
len(docs)

78

## VectorStore (stockage dans la base de donnée chromadb)

In [10]:
%%capture
! pip install langchain_chroma

In [1]:
import config

In [2]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

In [13]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large", api_key=secret)

vector_store = Chroma(
    collection_name="orange_data_source",
    embedding_function=embeddings,
    persist_directory="./chroma.db",
)

In [14]:
vector_store.add_documents(documents=docs)

['4281f652-5629-48c5-91ce-a608ebadfc5c',
 '8b65a2a4-4edd-455a-918a-eb3ac5290de8',
 '634e9064-1c7d-4f04-b326-bddfe64f2fe3',
 'ae9be9d3-58d5-4384-9c31-efb262986921',
 '15dd2c0b-b511-4bc3-9c12-c013aca9730f',
 'f0ced3b2-5e57-4c59-a97a-973a3043bb85',
 '51186624-1d3b-48f1-a527-70ea58cd9448',
 '8ed189df-795b-4571-8b84-492564d9e1c5',
 '4390ae1d-d8f8-4d92-9de1-164999a18131',
 '077424ae-fc6a-48ec-ba28-eeb1e921612f',
 '36d2b67b-ee83-4f90-82fd-2e5d1288028b',
 '55f3eef9-09c1-4de0-a84a-157714edad56',
 '9fd914a4-f2d1-49b2-a40a-4ec166728ef7',
 '641941ac-871f-46e0-b5b6-84b7ba3cb00b',
 '19ea886f-0b3f-49df-abb9-77a6442ff6f9',
 '12acbbb2-7e0b-4ff3-8358-d2e1ef5355cf',
 '907394db-58b6-409d-82af-a3488f1d4cc4',
 'd4ee9f5b-68d6-4d17-8667-6a837229db9a',
 'c39803f6-38a0-49b4-97a1-6b5426942a54',
 '0f09731c-ddbf-409b-a4d8-45b81bf5bb3f',
 'd72b0f90-c106-4567-892d-29a8e5b5b0b9',
 'd4d64792-e882-4616-9349-5fbea81048f2',
 '01add8be-408b-4d22-9f94-7e4edb9e1a5e',
 '5f1f9109-5223-4232-bb17-97f99a2b03a3',
 'c2d3cae6-10e5-

In [15]:
query="Comment débloquer mon compte Orange Money ?"

In [16]:
retriever = vector_store.as_retriever()

In [17]:
docs_similary=vector_store.similarity_search(query=query)

In [18]:
content=""""""
for idx,doc in enumerate(docs_similary):
  content += doc.page_content + f"\n\n{idx + 1} {doc.metadata['source']}\n\n"

In [39]:
from langchain.prompts import PromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

In [40]:
model=init_chat_model(
    model="gpt-4o",
    model_provider="openai",
    api_key=config.OPENAI_API_KEY
)

In [21]:
promt = """
  Tu est un assistant chez orange côte d'ivoire, reponds à la question de l'utilisteur
   selon le contexte suivant:
   te sera préciser.
   N'oublie pas de preciser la source ou tu as trouver la reponse au format Markdown.
<question>
{question}
<question>

<contexte>
{contexte}
<contexte>
"""

In [22]:
prompt_template=PromptTemplate.from_template(promt)

In [23]:
retriever=vector_store.as_retriever()

In [24]:
def retrieve_content(docs)->str:
  content=""""""
  for doc_id,doc in enumerate(docs):
    content += doc.page_content + f"\n\n{doc_id}"+ doc.metadata["source"]
  return content

In [25]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [26]:
chain={
    "contexte":retriever|retrieve_content,
    "question":RunnablePassthrough()
}|prompt_template|model|StrOutputParser()

In [27]:
answer=chain.invoke("Comment debloquer mon compte orange money")

Pour débloquer votre compte Orange Money en Côte d'Ivoire, vous pouvez utiliser le service de Selfcare de déblocage de compte OM. Ce service vous permet de débloquer votre compte via USSD ou l'application OMA. Si vous avez saisi incorrectement votre code secret Orange Money à cinq reprises, votre compte se bloque automatiquement, mais le service de Selfcare vous offre la possibilité de le débloquer.

Pour plus de détails ou pour accéder directement à la fonctionnalité de déblocage en un clic, vous pouvez consulter le lien suivant : [Comment débloquer mon compte Orange Money ? - Orange Côte d’Ivoire](https://www.orange.ci/fr/comment-debloquer-mon-compte-orange-money.html).

Source: [Orange Côte d’Ivoire - Comment débloquer mon compte Orange Money ?](https://www.orange.ci/fr/comment-debloquer-mon-compte-orange-money.html)

In [28]:
answer

'Pour débloquer votre compte Orange Money en Côte d\'Ivoire, vous pouvez suivre ces étapes :\n\n1. **Assurez-vous que vos informations de connexion sont correctes** : Vérifiez que vous avez saisi correctement votre numéro de mobile Orange ou votre adresse e-mail ainsi que votre mot de passe.\n\n2. **Réinitialisation du mot de passe** : Si vous avez oublié votre mot de passe, vous pouvez le réinitialiser en suivant les instructions sur la page de connexion d\'Orange. Cherchez l\'option "Mot de passe oublié" et suivez les étapes indiquées.\n\n3. **Contactez le service client d\'Orange** : Si vous ne pouvez toujours pas accéder à votre compte après avoir vérifié vos informations de connexion ou réinitialisé votre mot de passe, il est conseillé de contacter le service client d\'Orange Côte d\'Ivoire. Vous pouvez les joindre via leur ligne d\'assistance ou en visitant une de leurs agences.\n\nPour plus d\'informations, vous pouvez consulter la page officielle d\'Orange Côte d\'Ivoire [ici](

In [29]:
%pip install langchain_community

In [30]:
# pip install -U langchain langchain-community

from langchain_community.chat_models import ChatOpenAI
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain,create_history_aware_retriever
prompt_system = """Tu es un assistant service client chez orange côte d'ivoire,
ton rôle sera de repondre aux questions des utilisateurs en te basant sur
le contexte ci dessous :
<context>
{context}
</context>
Renvoie le résultat au format markdown"""



In [31]:
from langchain.prompts import MessagesPlaceholder

In [32]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", prompt_system),
        ("human","{input}")
     ]
)

In [33]:
chain = create_stuff_documents_chain(model, prompt)

In [34]:
docs = text_splitter.split_documents(docs_loader)

In [35]:
history_aware_retrieval = create_history_aware_retriever(model,retriever,prompt)

In [36]:
document = history_aware_retrieval.invoke({"chat_history":[{"role":"user","content":"C'est quoi l'app Orange et Moi"},

                                                {"role":"assistant","content":"""L'application **Orange et moi**
                                                est l'ancienne application d'Orange Côte d'Ivoire
                                                qui permettait aux utilisateurs de gérer leurs
                                                services mobiles et comptes Orange"""}],

                                                "input":"Comment fonctionne t-elle",
                                              "context":docs})

In [37]:
chain= create_stuff_documents_chain(model,prompt)

In [38]:
rag_chain = create_retrieval_chain(retriever,chain)

In [39]:
rag_chain.invoke({"input":"C'est quoi Max it"})

{'input': "C'est quoi Max it",
 'context': [Document(id='61e268bd-6df4-4ffc-a36a-d9715d177f27', metadata={'title': 'Max it, la super app qui fusionne Orange et moi et Orange Money ! | Orange Côte d’Ivoire', 'language': 'fr', 'description': 'Nouveau, la super app d’Orange vient de sortir ! C’est Max it, la fusion des applis Orange et moi avec Orange Money ! Achats de pass, transferts d’argent et paiement marchand avec Qr code, achats de tickets de concerts dans Marketplace, et plus encore…', 'split_number': 12, 'source': 'https://www.orange.ci/fr/max-it/super-app-orange-et-moi-orange-money.html'}, page_content="Qu'est ce que Max it ?\n Max it est une super app qui regroupe Orange et moi et Orange Money Afrique. Tu pourras y trouver tes services de communication, de fintech tels qu’Orange money ou Orange bank ou encore les partenaires Orange. Tu pourras acheter tes tickets d’événements, mais aussi gérer tes besoins quotidiens en nourriture, déplacements et bien plus encore..."),
  Docume

<réponse>

## Fonctionnalités de Max it

Max it, la super app proposée par Orange Côte d’Ivoire, fusionne les applications *Orange et moi* et *Orange Money* pour offrir une variété de fonctionnalités pratiques :

### Orange Money
- **Transferts d'argent** : Effectuer des transferts immédiats et sécurisés vers les comptes Orange Money.
- **Paiement de factures** : Faciliter le paiement de factures (eau, électricité, TV, internet, téléphones).
- **Paiement marchand** : Utilisation du QR code pour valider les achats en magasin.

### Ma Ligne (anciennement Orange et moi)
- **Achat de pass** : Sélectionner et acheter des pass mix, data, illimités ou spéciaux pour soi ou pour un proche.
- **Gestion de la consommation** : Consulter le solde et l'historique d'utilisation.
- **Promotions** : Accéder aux promos pour plus de volume data et minutes.

### Marketplace
- **Événements et Divertissements** : Acheter des tickets pour de concerts, festivals, et autres événements.
- **Musique et Jeux** : Accéder à des services de streaming musical et acheter des e-cartes pour les jeux vidéo populaires.

### Caractéristiques supplémentaires
- **Assistance 24/7** : Support continu pour aider les utilisateurs avec les nouvelles fonctionnalités.
- **Gestion internationale** : Utilisation de l'application pour les services internationaux.
- **Compatibilité étendue** : Disponible sur iOS (depuis la version 14), Android (depuis 5.0), et Huawei (version 5.1).

Max it garantit également aucune consommation de data tout en utilisant l'application, rendant l'expérience utilisateur sans frais supplémentaires liés à l'accès à Internet.

</réponse>

In [40]:

from langchain_community.chat_models import ChatOpenAI
from langchain.chains import create_history_aware_retriever
from langchain import hub

In [41]:
rephrase_prompt = hub.pull("langchain-ai/chat-langchain-rephrase",api_key=secret)

In [42]:
chat_retriever_chain = create_history_aware_retriever(
    model, retriever, rephrase_prompt
)


In [43]:
chat_retriever_chain

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x7de8b1426050>, search_kwargs={}))], default=PromptTemplate(input_variables=['chat_history', 'input'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'langchain-ai', 'lc_hub_repo': 'chat-langchain-rephrase', 'lc_hub_commit_hash': 'fb7ddb56be11b2ab10d176174dae36faa2a9a6ba13187c8b2b98315f6ca7d136'}, template='Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question.\n\nChat History:\n{chat_history}\nFollow Up Input: {input}\nStandalone Question:')
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x7de8b14c7d50>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7de8b1a216

In [44]:
%pip install -U beautifulsoup4

In [45]:
from langchain_community.document_loaders.recursive_url_loader import RecursiveUrlLoader

In [46]:
from langchain_community.document_loaders import RecursiveUrlLoader,WebBaseLoader

In [47]:
from bs4 import BeautifulSoup as Soup

In [48]:
%pip install requests

In [49]:
import requests

In [50]:
from langchain_core.documents import Document
from langchain.document_loaders import WebBaseLoader

In [51]:
urls_list =[
  {
    "source": "Ministère de la Construction, du Logement et de l'Urbanisme",
    "id": 1,
    "url": "https://construction.gouv.ci/faq/",
    "description": "La section Foire Aux Questions du Ministère de la Construction, du Logement et de l'Urbanisme, offrant des éclaircissements sur diverses thématiques"
  },
  {
    "source": "Ministère de la Construction, du Logement et de l'Urbanisme",
    "id": 2,
    "url": "https://construction.gouv.ci/lexique/",
    "description": "Lexique détaillé qui définit des termes cruciaux dans les domaines de l'habitat, de la construction, de la maintenance, du foncier et de l'urbanisme."
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 3,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/225/27",
    "description": "Comment demander un ACD à partir de l'attestation"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 4,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/226/27",
    "description": "Comment demander un ACD à partir de la lettre d'attribution"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 5,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/229/27",
    "description": "Demander un CERTIFICAT DE CONFORMITE DES LOTISSEMENTS"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 6,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/230/27",
    "description": "Demander un arrêté de concession définitive( ACD) à partir de la lettre d'attribution( LA)+l'arrêté de concession provisoire(ACP)+Titre Foncier(TF)"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 7,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/231/27",
    "description": "Comment demander une dérogation"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 8,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/232/27",
    "description": "Comment Demander un Déclassement et Morcellement"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 9,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/234/27",
    "description": "Demander un LOTISSEMENT VILLAGEOIS ABIDJAN ET YAMOUSSOUKRO (LVAY)"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 10,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/236/27",
    "description": "Comment Demander un Lotissement privé"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 11,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/237/27",
    "description": "Demander un Lotissement villageois intérieur du pays (LVIP)"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 12,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/238/27",
    "description": "Demander une Lettre d’attribution avec promesse de bail emphytéotique (LAPBE)"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 13,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/239/27",
    "description": "Demander un ACP avec promesse de bail emphytéotique (ACPPBE)"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 14,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/240/27",
    "description": "Demander un ACP avec promesse de bail emphytéotique AVEC TITRE FONCIER"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 15,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/241/27",
    "description": "Demander un Transfert d’ACP avec promesse de bail emphytéotique"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 16,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/240/27",
    "description": "Demander un ACP avec promesse de bail emphytéotique AVEC TITRE FONCIER"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 17,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/242/27",
    "description": "Bail emphytéotique"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 18,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/244/27",
    "description": "Demander un ACD hors lotissement"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 19,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/245/27",
    "description": "Demander une régularisation d'ACD"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 20,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/246/27",
    "description": "Demander une Attestation d’Attribution"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 21,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/247/27",
    "description": "Demander une position foncière"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 22,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/248/27",
    "description": "Demander une radiation de clause"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 23,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/249/27",
    "description": "Demander un Transfert de bail emphytéotique"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 24,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/254/27",
    "description": "Demander un ACD avec attestation (INTERIEUR DU PAYS )"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 25,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/256/27",
    "description": "Demander un ACD avec titre foncier (INTERIEUR DU PAYS )"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 26,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/257/27",
    "description": "Demander un ACD avec ACP (INTERIEUR DU PAYS )"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 27,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/363/27",
    "description": "Demander une consultations juridiques"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 28,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/370/27",
    "description": "Consolidation des droits concédés"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 29,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/472/27",
    "description": "Demander un état foncier / état historique"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 30,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/473/27",
    "description": "Demander un certificat de mutation de propriété foncière"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 31,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/473/27",
    "description": "Demander un certificat de mutation de propriété foncière"
  },
  {
    "source": "Minintère de la construction,de logement et de l'urbanisme",
    "id": 32,
    "url": "https://construction.gouv.ci/documents-administratifs-et-tarifs/#1504608467366-9d201e08-bf7b",
    "description": "Liste des procédures du guichet unique du permis de construire (GUPC)"
  },
  {
    "source": "Minintère de la construction,de logement et de l'urbanisme",
    "id": 33,
    "url": "https://construction.gouv.ci/documents-administratifs-et-tarifs/#1504608467429-ef352e24-1c07",
    "description": "Pièces à fournir et cout pour l’obtention d’un ACD"
  },
  {
    "source": "Agence Foncière rurale",
    "id": 34,
    "url": "https://www.afor.ci/",
    "description": "AFOR à propos"
  },
  {
    "source": "Agence Foncière rurale",
    "id": 35,
    "url": "https://www.afor.ci/procedures/delimitation-des-territoires-de-village",
    "description": "Délimitation des territoires de villages"
  },
  {
    "source": "Agence Foncière rurale",
    "id": 36,
    "url": "https://www.afor.ci/procedures/certificat-foncier",
    "description": "Délivrance du certificat foncier"
  },
  {
    "source": "Agence Foncière rurale",
    "id": 37,
    "url": "https://www.afor.ci/procedures/consolidation-des-droits-concedes",
    "description": "consolidation des droits concédés"
  },
  {
    "source": "Agence Foncière rurale",
    "id": 38,
    "url": "https://www.afor.ci/procedures/contractualisation",
    "description": "contractualisation"
  },
  {
    "source": "Agence Foncière rurale",
    "id": 39,
    "url": "https://www.afor.ci/procedures/titre-foncier",
    "description": "Titre foncier"
  }
]

In [52]:
urls_list1 =[
  {
    "source": "Ministère de la Construction, du Logement et de l'Urbanisme",
    "id": 1,
    "url": "https://construction.gouv.ci/faq/",
    "description": "La section Foire Aux Questions du Ministère de la Construction, du Logement et de l'Urbanisme, offrant des éclaircissements sur diverses thématiques"
  },
  {
    "source": "Ministère de la Construction, du Logement et de l'Urbanisme",
    "id": 2,
    "url": "https://construction.gouv.ci/lexique/",
    "description": "Lexique détaillé qui définit des termes cruciaux dans les domaines de l'habitat, de la construction, de la maintenance, du foncier et de l'urbanisme."
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 3,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/225/27",
    "description": "Comment demander un ACD à partir de l'attestation"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 4,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/226/27",
    "description": "Comment demander un ACD à partir de la lettre d'attribution"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 5,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/229/27",
    "description": "Demander un CERTIFICAT DE CONFORMITE DES LOTISSEMENTS"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 6,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/230/27",
    "description": "Demander un arrêté de concession définitive( ACD) à partir de la lettre d'attribution( LA)+l'arrêté de concession provisoire(ACP)+Titre Foncier(TF)"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 7,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/231/27",
    "description": "Comment demander une dérogation"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 8,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/232/27",
    "description": "Comment Demander un Déclassement et Morcellement"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 9,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/234/27",
    "description": "Demander un LOTISSEMENT VILLAGEOIS ABIDJAN ET YAMOUSSOUKRO (LVAY)"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 10,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/236/27",
    "description": "Comment Demander un Lotissement privé"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 11,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/237/27",
    "description": "Demander un Lotissement villageois intérieur du pays (LVIP)"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 12,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/238/27",
    "description": "Demander une Lettre d’attribution avec promesse de bail emphytéotique (LAPBE)"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 13,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/239/27",
    "description": "Demander un ACP avec promesse de bail emphytéotique (ACPPBE)"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 14,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/240/27",
    "description": "Demander un ACP avec promesse de bail emphytéotique AVEC TITRE FONCIER"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 15,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/241/27",
    "description": "Demander un Transfert d’ACP avec promesse de bail emphytéotique"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 16,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/240/27",
    "description": "Demander un ACP avec promesse de bail emphytéotique AVEC TITRE FONCIER"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 17,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/242/27",
    "description": "Bail emphytéotique"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 18,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/244/27",
    "description": "Demander un ACD hors lotissement"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 19,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/245/27",
    "description": "Demander une régularisation d'ACD"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 20,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/246/27",
    "description": "Demander une Attestation d’Attribution"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 21,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/247/27",
    "description": "Demander une position foncière"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 22,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/248/27",
    "description": "Demander une radiation de clause"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 23,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/249/27",
    "description": "Demander un Transfert de bail emphytéotique"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 24,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/254/27",
    "description": "Demander un ACD avec attestation (INTERIEUR DU PAYS )"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 25,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/256/27",
    "description": "Demander un ACD avec titre foncier (INTERIEUR DU PAYS )"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 26,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/257/27",
    "description": "Demander un ACD avec ACP (INTERIEUR DU PAYS )"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 27,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/363/27",
    "description": "Demander une consultations juridiques"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 28,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/370/27",
    "description": "Consolidation des droits concédés"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 29,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/472/27",
    "description": "Demander un état foncier / état historique"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 30,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/473/27",
    "description": "Demander un certificat de mutation de propriété foncière"
  },
  {
    "source": "servicepublic.gouv.ci",
    "id": 31,
    "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/473/27",
    "description": "Demander un certificat de mutation de propriété foncière"
  },
  {
    "source": "Minintère de la construction,de logement et de l'urbanisme",
    "id": 32,
    "url": "https://construction.gouv.ci/documents-administratifs-et-tarifs/#1504608467366-9d201e08-bf7b",
    "description": "Liste des procédures du guichet unique du permis de construire (GUPC)"
  },
  {
    "source": "Minintère de la construction,de logement et de l'urbanisme",
    "id": 33,
    "url": "https://construction.gouv.ci/documents-administratifs-et-tarifs/#1504608467429-ef352e24-1c07",
    "description": "Pièces à fournir et cout pour l’obtention d’un ACD"
  },
  {
    "source": "Agence Foncière rurale",
    "id": 34,
    "url": "https://www.afor.ci/",
    "description": "AFOR à propos"
  },
  {
    "source": "Agence Foncière rurale",
    "id": 35,
    "url": "https://www.afor.ci/procedures/delimitation-des-territoires-de-village",
    "description": "Délimitation des territoires de villages"
  },
  {
    "source": "Agence Foncière rurale",
    "id": 36,
    "url": "https://www.afor.ci/procedures/certificat-foncier",
    "description": "Délivrance du certificat foncier"
  },
  {
    "source": "Agence Foncière rurale",
    "id": 37,
    "url": "https://www.afor.ci/procedures/consolidation-des-droits-concedes",
    "description": "consolidation des droits concédés"
  },
  {
    "source": "Agence Foncière rurale",
    "id": 38,
    "url": "https://www.afor.ci/procedures/contractualisation",
    "description": "contractualisation"
  },
  {
    "source": "Agence Foncière rurale",
    "id": 39,
    "url": "https://www.afor.ci/procedures/titre-foncier",
    "description": "Titre foncier"
  }
]

In [53]:
for i in range(len(urls_list1)):
  del urls_list1[i]["id"]

In [54]:
import json

In [55]:
urls_list2 = json.dumps(urls_list1,ensure_ascii=False)
(urls_list2)

'[{"source": "Ministère de la Construction, du Logement et de l\'Urbanisme", "url": "https://construction.gouv.ci/faq/", "description": "La section Foire Aux Questions du Ministère de la Construction, du Logement et de l\'Urbanisme, offrant des éclaircissements sur diverses thématiques"}, {"source": "Ministère de la Construction, du Logement et de l\'Urbanisme", "url": "https://construction.gouv.ci/lexique/", "description": "Lexique détaillé qui définit des termes cruciaux dans les domaines de l\'habitat, de la construction, de la maintenance, du foncier et de l\'urbanisme."}, {"source": "servicepublic.gouv.ci", "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/225/27", "description": "Comment demander un ACD à partir de l\'attestation"}, {"source": "servicepublic.gouv.ci", "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/226/27", "description": "Comment demander un ACD à partir de la lettre d\'attribution"}, {"source": "servicepublic.go

In [56]:

[
    {
        "source": "Ministère de la Construction, du Logement et de l'Urbanisme",
        "url": "https://construction.gouv.ci/faq/",
        "description": "La section Foire Aux Questions du Ministère de la Construction, du Logement et de l'Urbanisme, offrant des éclaircissements sur diverses thématiques"
    },
    {
        "source": "Ministère de la Construction, du Logement et de l'Urbanisme",
        "url": "https://construction.gouv.ci/lexique/",
        "description": "Lexique détaillé qui définit des termes cruciaux dans les domaines de l'habitat, de la construction, de la maintenance, du foncier et de l'urbanisme."
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/225/27",
        "description": "Comment demander un ACD à partir de l'attestation"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/226/27",
        "description": "Comment demander un ACD à partir de la lettre d'attribution"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/229/27",
        "description": "Demander un CERTIFICAT DE CONFORMITE DES LOTISSEMENTS"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/230/27",
        "description": "Demander un arrêté de concession définitive( ACD) à partir de la lettre d'attribution( LA)+l'arrêté de concession provisoire(ACP)+Titre Foncier(TF)"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/231/27",
        "description": "Comment demander une dérogation"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/232/27",
        "description": "Comment Demander un Déclassement et Morcellement"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/234/27",
        "description": "Demander un LOTISSEMENT VILLAGEOIS ABIDJAN ET YAMOUSSOUKRO (LVAY)"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/236/27",
        "description": "Comment Demander un Lotissement privé"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/237/27",
        "description": "Demander un Lotissement villageois intérieur du pays (LVIP)"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/238/27",
        "description": "Demander une Lettre d’attribution avec promesse de bail emphytéotique (LAPBE)"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/239/27",
        "description": "Demander un ACP avec promesse de bail emphytéotique (ACPPBE)"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/240/27",
        "description": "Demander un ACP avec promesse de bail emphytéotique AVEC TITRE FONCIER"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/241/27",
        "description": "Demander un Transfert d’ACP avec promesse de bail emphytéotique"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/240/27",
        "description": "Demander un ACP avec promesse de bail emphytéotique AVEC TITRE FONCIER"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/242/27",
        "description": "Bail emphytéotique"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/244/27",
        "description": "Demander un ACD hors lotissement"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/245/27",
        "description": "Demander une régularisation d'ACD"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/246/27",
        "description": "Demander une Attestation d’Attribution"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/247/27",
        "description": "Demander une position foncière"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/248/27",
        "description": "Demander une radiation de clause"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/249/27",
        "description": "Demander un Transfert de bail emphytéotique"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/254/27",
        "description": "Demander un ACD avec attestation (INTERIEUR DU PAYS )"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/256/27",
        "description": "Demander un ACD avec titre foncier (INTERIEUR DU PAYS )"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/257/27",
        "description": "Demander un ACD avec ACP (INTERIEUR DU PAYS )"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/363/27",
        "description": "Demander une consultations juridiques"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/370/27",
        "description": "Consolidation des droits concédés"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/472/27",
        "description": "Demander un état foncier / état historique"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/473/27",
        "description": "Demander un certificat de mutation de propriété foncière"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/473/27",
        "description": "Demander un certificat de mutation de propriété foncière"
    },
    {
        "source": "Minintère de la construction,de logement et de l'urbanisme",
        "url": "https://construction.gouv.ci/documents-administratifs-et-tarifs/#1504608467366-9d201e08-bf7b",
        "description": "Liste des procédures du guichet unique du permis de construire (GUPC)"
    },
    {
        "source": "Minintère de la construction,de logement et de l'urbanisme",
        "url": "https://construction.gouv.ci/documents-administratifs-et-tarifs/#1504608467429-ef352e24-1c07",
        "description": "Pièces à fournir et cout pour l’obtention d’un ACD"
    },
    {
        "source": "Agence Foncière rurale",
        "url": "https://www.afor.ci/",
        "description": "AFOR à propos"
    },
    {
        "source": "Agence Foncière rurale",
        "url": "https://www.afor.ci/procedures/delimitation-des-territoires-de-village",
        "description": "Délimitation des territoires de villages"
    },
    {
        "source": "Agence Foncière rurale",
        "url": "https://www.afor.ci/procedures/certificat-foncier",
        "description": "Délivrance du certificat foncier"
    },
    {
        "source": "Agence Foncière rurale",
        "url": "https://www.afor.ci/procedures/consolidation-des-droits-concedes",
        "description": "consolidation des droits concédés"
    },
    {
        "source": "Agence Foncière rurale",
        "url": "https://www.afor.ci/procedures/contractualisation",
        "description": "contractualisation"
    },
    {
        "source": "Agence Foncière rurale",
        "url": "https://www.afor.ci/procedures/titre-foncier",
        "description": "Titre foncier"
    }
]

[{'source': "Ministère de la Construction, du Logement et de l'Urbanisme",
  'url': 'https://construction.gouv.ci/faq/',
  'description': "La section Foire Aux Questions du Ministère de la Construction, du Logement et de l'Urbanisme, offrant des éclaircissements sur diverses thématiques"},
 {'source': "Ministère de la Construction, du Logement et de l'Urbanisme",
  'url': 'https://construction.gouv.ci/lexique/',
  'description': "Lexique détaillé qui définit des termes cruciaux dans les domaines de l'habitat, de la construction, de la maintenance, du foncier et de l'urbanisme."},
 {'source': 'servicepublic.gouv.ci',
  'url': 'https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/225/27',
  'description': "Comment demander un ACD à partir de l'attestation"},
 {'source': 'servicepublic.gouv.ci',
  'url': 'https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/226/27',
  'description': "Comment demander un ACD à partir de la lettre d'attribution"},
 {'source': 'serv

In [57]:
def add_contact_info(urls : list[dict])->list[dict]:

  for i in range(len(urls)):

    if urls[i]["source"] == "Ministère de la Construction, du Logement et de l'Urbanisme":
      urls[i]["email"] = "centredappelmclu@construction.gouv.ci / scpcimclu@construction.gouv.ci"
      urls[i]["siege"] = "Plateau, Tour A, 16e - 17e étage BP V 153 Abidjan"
      urls[i]["telephone"] = "07 07 56 45 45 / 01 40 22 78 78"

    elif urls[i]["source"] == "servicepublic.gouv.ci":
      urls[i]["email"] = "osep@modernisation.gouv.ci"
      urls[i]["siege"] = ""
      urls[i]["telephone"] = "+225 22 40 98 98"

    elif urls[i]["source"] == "Agence Foncière rurale":
      urls[i]["email"] = "infos@afor.ci"
      urls[i]["siege"] = "Abidjan, Cocody-Angré 7è Tranche Quartier Zinsou 1, Rue L 183"
      urls[i]["telephone"] = "+225 2722505171 / +225 0798737398"

      return urls

In [58]:
urls_list2 =  add_contact_info(urls_list1)

In [59]:
urls_list2 = json.dumps(urls_list2 , indent =4 , ensure_ascii = False)

In [60]:
print(urls_list2)

[
    {
        "source": "Ministère de la Construction, du Logement et de l'Urbanisme",
        "url": "https://construction.gouv.ci/faq/",
        "description": "La section Foire Aux Questions du Ministère de la Construction, du Logement et de l'Urbanisme, offrant des éclaircissements sur diverses thématiques",
        "email": "centredappelmclu@construction.gouv.ci / scpcimclu@construction.gouv.ci",
        "siege": "Plateau, Tour A, 16e - 17e étage BP V 153 Abidjan",
        "telephone": "07 07 56 45 45 / 01 40 22 78 78"
    },
    {
        "source": "Ministère de la Construction, du Logement et de l'Urbanisme",
        "url": "https://construction.gouv.ci/lexique/",
        "description": "Lexique détaillé qui définit des termes cruciaux dans les domaines de l'habitat, de la construction, de la maintenance, du foncier et de l'urbanisme.",
        "email": "centredappelmclu@construction.gouv.ci / scpcimclu@construction.gouv.ci",
        "siege": "Plateau, Tour A, 16e - 17e étage 

#### Debut de L'indexation

In [52]:
urls_last

['https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/226/27',
 'https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/230/27',
 'https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/231/27',
 'https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/234/27',
 'https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/237/27',
 'https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/238/27',
 'https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/240/27',
 'https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/241/27',
 'https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/243/27',
 'https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/245/27',
 'https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/246/27',
 'https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/248/27',
 'https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/254/27',

In [1]:
urls_to_docs =[
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/225/27",
        "description": "Comment demander un ACD à partir de l'attestation",
        "email": "osep@modernisation.gouv.ci",
        "siege": "",
        "telephone": "+225 22 40 98 98"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/226/27",
        "description": "Comment demander un ACD à partir de la lettre d'attribution",
        "email": "osep@modernisation.gouv.ci",
        "siege": "",
        "telephone": "+225 22 40 98 98"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/229/27",
        "description": "Demander un CERTIFICAT DE CONFORMITE DES LOTISSEMENTS",
        "email": "osep@modernisation.gouv.ci",
        "siege": "",
        "telephone": "+225 22 40 98 98"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/230/27",
        "description": "Demander un arrêté de concession définitive( ACD) à partir de la lettre d'attribution( LA)+l'arrêté de concession provisoire(ACP)+Titre Foncier(TF)",
        "email": "osep@modernisation.gouv.ci",
        "siege": "",
        "telephone": "+225 22 40 98 98"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/231/27",
        "description": "Comment demander une dérogation",
        "email": "osep@modernisation.gouv.ci",
        "siege": "",
        "telephone": "+225 22 40 98 98"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/232/27",
        "description": "Comment Demander un Déclassement et Morcellement",
        "email": "osep@modernisation.gouv.ci",
        "siege": "",
        "telephone": "+225 22 40 98 98"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/234/27",
        "description": "Demander un LOTISSEMENT VILLAGEOIS ABIDJAN ET YAMOUSSOUKRO (LVAY)",
        "email": "osep@modernisation.gouv.ci",
        "siege": "",
        "telephone": "+225 22 40 98 98"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/236/27",
        "description": "Comment Demander un Lotissement privé",
        "email": "osep@modernisation.gouv.ci",
        "siege": "",
        "telephone": "+225 22 40 98 98"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/237/27",
        "description": "Demander un Lotissement villageois intérieur du pays (LVIP)",
        "email": "osep@modernisation.gouv.ci",
        "siege": "",
        "telephone": "+225 22 40 98 98"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/238/27",
        "description": "Demander une Lettre d’attribution avec promesse de bail emphytéotique (LAPBE)",
        "email": "osep@modernisation.gouv.ci",
        "siege": "",
        "telephone": "+225 22 40 98 98"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/239/27",
        "description": "Demander un ACP avec promesse de bail emphytéotique (ACPPBE)",
        "email": "osep@modernisation.gouv.ci",
        "siege": "",
        "telephone": "+225 22 40 98 98"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/240/27",
        "description": "Demander un ACP avec promesse de bail emphytéotique AVEC TITRE FONCIER",
        "email": "osep@modernisation.gouv.ci",
        "siege": "",
        "telephone": "+225 22 40 98 98"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/241/27",
        "description": "Demander un Transfert d’ACP avec promesse de bail emphytéotique",
        "email": "osep@modernisation.gouv.ci",
        "siege": "",
        "telephone": "+225 22 40 98 98"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/240/27",
        "description": "Demander un ACP avec promesse de bail emphytéotique AVEC TITRE FONCIER",
        "email": "osep@modernisation.gouv.ci",
        "siege": "",
        "telephone": "+225 22 40 98 98"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/242/27",
        "description": "Bail emphytéotique",
        "email": "osep@modernisation.gouv.ci",
        "siege": "",
        "telephone": "+225 22 40 98 98"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/244/27",
        "description": "Demander un ACD hors lotissement",
        "email": "osep@modernisation.gouv.ci",
        "siege": "",
        "telephone": "+225 22 40 98 98"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/245/27",
        "description": "Demander une régularisation d'ACD",
        "email": "osep@modernisation.gouv.ci",
        "siege": "",
        "telephone": "+225 22 40 98 98"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/246/27",
        "description": "Demander une Attestation d’Attribution",
        "email": "osep@modernisation.gouv.ci",
        "siege": "",
        "telephone": "+225 22 40 98 98"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/247/27",
        "description": "Demander une position foncière",
        "email": "osep@modernisation.gouv.ci",
        "siege": "",
        "telephone": "+225 22 40 98 98"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/248/27",
        "description": "Demander une radiation de clause",
        "email": "osep@modernisation.gouv.ci",
        "siege": "",
        "telephone": "+225 22 40 98 98"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/249/27",
        "description": "Demander un Transfert de bail emphytéotique",
        "email": "osep@modernisation.gouv.ci",
        "siege": "",
        "telephone": "+225 22 40 98 98"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/254/27",
        "description": "Demander un ACD avec attestation (INTERIEUR DU PAYS )",
        "email": "osep@modernisation.gouv.ci",
        "siege": "",
        "telephone": "+225 22 40 98 98"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/256/27",
        "description": "Demander un ACD avec titre foncier (INTERIEUR DU PAYS )",
        "email": "osep@modernisation.gouv.ci",
        "siege": "",
        "telephone": "+225 22 40 98 98"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/257/27",
        "description": "Demander un ACD avec ACP (INTERIEUR DU PAYS )",
        "email": "osep@modernisation.gouv.ci",
        "siege": "",
        "telephone": "+225 22 40 98 98"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/363/27",
        "description": "Demander une consultations juridiques",
        "email": "osep@modernisation.gouv.ci",
        "siege": "",
        "telephone": "+225 22 40 98 98"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/370/27",
        "description": "Consolidation des droits concédés",
        "email": "osep@modernisation.gouv.ci",
        "siege": "",
        "telephone": "+225 22 40 98 98"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/472/27",
        "description": "Demander un état foncier / état historique",
        "email": "osep@modernisation.gouv.ci",
        "siege": "",
        "telephone": "+225 22 40 98 98"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/473/27",
        "description": "Demander un certificat de mutation de propriété foncière",
        "email": "osep@modernisation.gouv.ci",
        "siege": "",
        "telephone": "+225 22 40 98 98"
    },
    {
        "source": "servicepublic.gouv.ci",
        "url": "https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/473/27",
        "description": "Demander un certificat de mutation de propriété foncière",
        "email": "osep@modernisation.gouv.ci",
        "siege": "",
        "telephone": "+225 22 40 98 98"
    },
    {
        "source": "Agence Foncière rurale",
        "url": "https://www.afor.ci/procedures/delimitation-des-territoires-de-village",
        "description": "Délimitation des territoires de villages",
        "email": "infos@afor.ci",
        "siege": "Abidjan, Cocody-Angré 7è Tranche Quartier Zinsou 1, Rue L 183",
        "telephone": "+225 2722505171 / +225 0798737398"
    },
    {
        "source": "Agence Foncière rurale",
        "url": "https://www.afor.ci/procedures/certificat-foncier",
        "description": "Délivrance du certificat foncier",
        "email": "infos@afor.ci",
        "siege": "Abidjan, Cocody-Angré 7è Tranche Quartier Zinsou 1, Rue L 183",
        "telephone": "+225 2722505171 / +225 0798737398"
    },
    {
        "source": "Agence Foncière rurale",
        "url": "https://www.afor.ci/procedures/consolidation-des-droits-concedes",
        "description": "consolidation des droits concédés",
        "email": "infos@afor.ci",
        "siege": "Abidjan, Cocody-Angré 7è Tranche Quartier Zinsou 1, Rue L 183",
        "telephone": "+225 2722505171 / +225 0798737398"
    },
    {
        "source": "Agence Foncière rurale",
        "url": "https://www.afor.ci/procedures/contractualisation",
        "description": "contractualisation",
        "email": "infos@afor.ci",
        "siege": "Abidjan, Cocody-Angré 7è Tranche Quartier Zinsou 1, Rue L 183",
        "telephone": "+225 2722505171 / +225 0798737398"
    },
    {
        "source": "Agence Foncière rurale",
        "url": "https://www.afor.ci/procedures/titre-foncier",
        "description": "Titre foncier",
        "email": "infos@afor.ci",
        "siege": "Abidjan, Cocody-Angré 7è Tranche Quartier Zinsou 1, Rue L 183",
        "telephone": "+225 2722505171 / +225 0798737398"
    },
    {
        "source": "Agence Foncière rurale",
        "url": "https://www.afor.ci/",
        "description": "AFOR à propos",
        "email": "infos@afor.ci",
        "siege": "Abidjan, Cocody-Angré 7è Tranche Quartier Zinsou 1, Rue L 183",
        "telephone": "+225 2722505171 / +225 0798737398"
    },
    {
    "source": "Ministère de la Construction, du Logement et de l'urbanisme",
    "url": "https://construction.gouv.ci/ordre-des-architectes/",
    "description": "Ordre des architectes",
    "email":"architectes@cnoa.ci",
    "siege": "Immeuble Carbonne",
    "telephone": "+225 27 22 48 50 38"
  },
   {
    "source": "Ministère de la Construction, du Logement et de l'urbanisme",
    "url": "https://construction.gouv.ci/ordre-des-geometres/",
    "description": "Ordre des geometres",
    "email":" infos@geometre-expert.ci",
    "siege": "Abidjan Cocody, Lycée Technique, Rue B23, Batiment V2, 2e étage, porte 364",
    "telephone": "+225 27 22 44 96 94/ +225 27 22 48 75 29/ +225 07 07 78 77 77"
  },
    {
    "source": "Ministère de la Construction, du Logement et de l'urbanisme",
    "url": "https://construction.gouv.ci/ordre-des-urbanistes/",
    "description": "Ordre des urbanistes",
    "email":"ordre.urbanistes.ci@gmail.com",
    "siege": "08 BP 1117 ABIDJAN, 08",
    "telephone": "(+225) 27 22 48 77 07"
  },
    {
    "source": "Ministère de la Construction, du Logement et de l'urbanisme",
    "url": "https://construction.gouv.ci/chambre-des-notaires/",
    "description": "Ordre des notaires",
    "email":"chambre@notaire.ci",
    "siege":"Avenue Delafosse prolongée Immeuble Horizon 1er étage",
    "telephone":"+225 20 32 11 47 / +225 07 06 17 16"
  },
   {
    "source": "Ministère de la Construction, du Logement et de l'urbanisme",
    "url": "https://construction.gouv.ci/liste-des-promoteurs-agrees/",
    "description": "Liste des promoteurs immobiliers agrées",
    "email": "centredappelmclu@construction.gouv.ci / scpcimclu@construction.gouv.ci",
    "siege": "Plateau, Tour A, 16e - 17e étage BP V 153 Abidjan",
    "telephone": "07 07 56 45 45 / 01 40 22 78 78"
  },
  {
    "source": "Ministère de la Construction, du Logement et de l'urbanisme",
    "url": "https://construction.gouv.ci/amenageurs-fonciers-agrees/",
    "description": "Aménageurs fonciers agrées",
    "email": "centredappelmclu@construction.gouv.ci / scpcimclu@construction.gouv.ci",
    "siege": "Plateau, Tour A, 16e - 17e étage BP V 153 Abidjan",
    "telephone": "07 07 56 45 45 / 01 40 22 78 78"
  },
   {
        "source": "Ministère de la Construction, du Logement et de l'urbanisme",
        "url": "https://construction.gouv.ci/faq/",
        "description": "La section Foire Aux Questions du Ministère de la Construction, du Logement et de l'Urbanisme, offrant des éclaircissements sur diverses thématiques",
        "email": "centredappelmclu@construction.gouv.ci / scpcimclu@construction.gouv.ci",
        "siege": "Plateau, Tour A, 16e - 17e étage BP V 153 Abidjan",
        "telephone": "07 07 56 45 45 / 01 40 22 78 78"
    },
    {
        "source": "Ministère de la Construction, du Logement et de l'urbanisme",
        "url": "https://construction.gouv.ci/lexique/",
        "description": "Lexique détaillé qui définit des termes cruciaux dans les domaines de l'habitat, de la construction, de la maintenance, du foncier et de l'urbanisme.",
        "email": "centredappelmclu@construction.gouv.ci / scpcimclu@construction.gouv.ci",
        "siege": "Plateau, Tour A, 16e - 17e étage BP V 153 Abidjan",
        "telephone": "07 07 56 45 45 / 01 40 22 78 78"
    },
        {
        "source": "Ministère de la Construction, du Logement et de l'urbanisme",
        "url": "https://construction.gouv.ci/documents-administratifs-et-tarifs/#1504608467366-9d201e08-bf7b",
        "description": "Liste des procédures du guichet unique du permis de construire (GUPC)",
        "email": "centredappelmclu@construction.gouv.ci / scpcimclu@construction.gouv.ci",
        "siege": "Plateau, Tour A, 16e - 17e étage BP V 153 Abidjan",
        "telephone": "07 07 56 45 45 / 01 40 22 78 78"
    },
    {
        "source": "Ministère de la Construction, du Logement et de l'urbanisme",
        "url": "https://construction.gouv.ci/documents-administratifs-et-tarifs/#1504608467429-ef352e24-1c07",
        "description": "Pièces à fournir et cout pour l’obtention d’un ACD",
        "email": "centredappelmclu@construction.gouv.ci / scpcimclu@construction.gouv.ci",
        "siege": "Plateau, Tour A, 16e - 17e étage BP V 153 Abidjan",
        "telephone": "07 07 56 45 45 / 01 40 22 78 78"
    },
]

In [3]:
from langchain_core.documents import Document

In [4]:
def create_docs_with_urls(urls_list : list[dict])->Document:
  docs = []
  for url in urls_list:
      loader = WebBaseLoader(url["url"])
      docs.append(loader.load()[0])

  return docs

In [5]:
def add_new_metada(docs : list[Document] , urls : list[dict]):
    """Add some metada to the """
    final_docs = []
    for i,doc in enumerate(docs):
        doc.metadata["email"] = urls[i]["email"]
        doc.metadata["telephone"] = urls[i]["telephone"]
        doc.metadata["siege"] = urls[i]["siege"]
        doc.metadata["description"] = urls[i]["description"]
        doc.metadata["provenance"] = urls[i]["source"]
        final_docs.append(doc)

    return final_docs

### FINAL URL DOCUMENT

In [6]:
from langchain_community.document_loaders import WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [7]:
docs = create_docs_with_urls(urls_to_docs)

In [8]:
final_docs = add_new_metada(docs , urls_to_docs)

## CREATE DOCUMENT LOADER PATH

In [5]:
!python -m pip install --upgrade pip

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----------- ---------------------------- 0.5/1.8 MB 580.3 kB/s eta 0:00:03
   ----------- ---------------------------- 0.5/1.8 MB 580.3 kB/s eta 0:00:03
   ----------------------- ---------------- 1.0/1.8 MB 802.3 kB/s eta 0:00:01
   ----------------------- ---------------- 1.0/1.8 MB 802.3 kB/s eta 0:00:01
   ----------------------- ---------------- 1.0/1.8 MB 802.3 kB/s eta 0:00:01
   ----------------------------- ---------- 1.3/1.8 MB 714.6 kB/s eta 0:00:01
   ----------------------------------- ---- 1.6/1.8 MB 772.3 kB/s eta 0:00:01
   ----------------------------------- ---- 1.6/1.8 MB 772.3 kB/s eta 0:00:01
   ----------------------------

In [9]:
%pip install -qU pypdf

Note: you may need to restart the kernel to use updated packages.


In [74]:
from langchain_community.document_loaders import PyPDFLoader

c:\Users\HP\Desktop\AI\GENAI\PROJET FORMATION\Projet Final\Assistant_foncier\api_llm


WindowsPath('c:/Users/HP/Desktop/AI/GENAI/PROJET FORMATION/Projet Final/Assistant_foncier/api_llm')

In [10]:
import config

In [11]:
from langchain_community.document_loaders.pdf import PyPDFDirectoryLoader

In [12]:
file_directory = config.PDF_DIR

In [13]:
print(file_directory)

c:\Users\HP\Desktop\AI\GENAI\PROJET FORMATION\Projet Final\Assistant_foncier\api_llm\documents_pdf


In [14]:
pdf_loader = PyPDFDirectoryLoader(path = file_directory , mode = "page")

In [81]:
type(pdf_loader)

langchain_community.document_loaders.pdf.PyPDFDirectoryLoader

In [15]:
pdf_docs = pdf_loader.load()

In [85]:
len(pdf_docs)

336

In [30]:
len(pdf_docs)

0

In [86]:
final_docs[0].metadata

{'source': 'https://servicepublic.gouv.ci/accueil/detaildemarcheparticulier/1/225/27',
 'title': "Service Public de Côte d'Ivoire :: servicepublic.gouv.ci",
 'language': 'No language found.',
 'email': 'osep@modernisation.gouv.ci',
 'telephone': '+225 22 40 98 98',
 'siege': '',
 'description': "Comment demander un ACD à partir de l'attestation",
 'provenance': 'servicepublic.gouv.ci'}

In [199]:
pdf_docs[6].metadata

{'producer': '',
 'creator': '',
 'creationdate': '',
 'author': '',
 'title': '',
 'subject': '',
 'source': '/content/drive/MyDrive/documents_loads/documents_pdf/Arrêté-interm.-n°-567-du-16-07-2021-rapportant-indemnités-éviction.pdf',
 'total_pages': 4,
 'page': 0,
 'page_label': '1'}

In [16]:
all_docs = []
for doc in docs:
    all_docs.append(doc)

In [17]:
len(all_docs)

45

In [18]:
for doc in pdf_docs:
    all_docs.append(doc)

In [99]:
len(all_docs)

381

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter,CharacterTextSplitter

In [31]:
docs_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    encoding_name = "cl100k_base",
    chunk_size = 100,
    chunk_overlap = 10)

In [32]:
docs_split = docs_splitter.split_documents(documents = all_docs)

Created a chunk of size 161, which is longer than the specified 100
Created a chunk of size 407, which is longer than the specified 100
Created a chunk of size 130, which is longer than the specified 100
Created a chunk of size 193, which is longer than the specified 100
Created a chunk of size 105, which is longer than the specified 100
Created a chunk of size 110, which is longer than the specified 100


Created a chunk of size 111, which is longer than the specified 100
Created a chunk of size 300, which is longer than the specified 100
Created a chunk of size 225, which is longer than the specified 100
Created a chunk of size 236, which is longer than the specified 100
Created a chunk of size 208, which is longer than the specified 100
Created a chunk of size 198, which is longer than the specified 100
Created a chunk of size 108, which is longer than the specified 100
Created a chunk of size 168, which is longer than the specified 100
Created a chunk of size 563, which is longer than the specified 100
Created a chunk of size 453, which is longer than the specified 100
Created a chunk of size 209, which is longer than the specified 100
Created a chunk of size 197, which is longer than the specified 100
Created a chunk of size 106, which is longer than the specified 100
Created a chunk of size 132, which is longer than the specified 100
Created a chunk of size 104, which is longer tha

In [101]:
len(docs_split)

773

In [25]:
from langchain_openai import OpenAIEmbeddings

In [26]:
model_embedding = OpenAIEmbeddings(model = "text-embedding-3-large", api_key = config.OPENAI_API_KEY)

In [27]:
from langchain_chroma import Chroma

In [108]:
vectorstore = Chroma.from_documents(docs_split , model_embedding , persist_directory = config.PERSIST_DIR)

In [28]:
vectorstore= Chroma(
    collection_name = "final_chroma_db_database",
    persist_directory = config.PERSIST_DIR,
    embedding_function = model_embedding
)

In [ ]:
vectorstore.add_documents(documents = docs_split)

['d03ebff1-c24f-47b1-be2c-bf4f406af96d',
 'fcb7b0c2-f333-4b08-92ea-3dd09ce0109f',
 '0adb1b75-118e-4ffe-ae64-21c3675cc09f',
 'ffb58b0d-724a-498c-9eee-90a0489be296',
 'c6c86276-37bc-4191-b8ee-a95ad37b225c',
 '8d9f4bef-3ebc-40b1-8822-700f05adda15',
 '72d9b2e5-9de1-4362-aa2b-ed3e63885f04',
 '03a7fc34-59ef-4f3c-9311-72320fcff8ca',
 '4552038e-f83b-4a26-8e5a-11a61016a334',
 'd4a9c775-bf70-4754-af5e-d15389e14bb2',
 '46cda0de-9843-473a-974f-105ef57094b5',
 '84f53547-1a81-40be-9383-bb85bfdd4923',
 'a2fb9af0-de53-4bad-91e0-093440d7d7f5',
 '56d8b9bf-862b-4780-93ff-25b6c5a88175',
 'e51d9f1b-e012-4c63-91cb-f15d9a561a09',
 '0e3ce551-1b55-4aff-abed-5cbabd2fa709',
 '5bfcfa1f-8091-461c-b04a-782610f8a6ba',
 'aae138f9-54d7-4f97-b85d-09d0a13ce659',
 '1607ae1f-a52e-438a-8b3d-cd735b8f3e12',
 '558b531e-5350-4162-8aab-e4b0a62dc165',
 '1ea10494-a91c-430c-bf5f-971786556953',
 'ff4e60c3-ed4b-4154-9463-f9c817b2b9c7',
 'f782534b-b680-4506-9952-fa08b79a93cc',
 '1efd20ec-6a3d-45e2-9bed-aa7f0c90917a',
 'ff64258f-bd32-

In [85]:
prompt = """
Tu est un assistant au service du Ministere de la construction et du logement
de Côte d'ivoire, ton rôle sera de repondre aux questions des
utilisateur en te basant sur le conteste suivant:
voici le contexte:
<contexte>
{contexte}
</contexte>
voici la question:
<question>
{question}
<question>
Renvoie le le resultat au format markdown
n'hésite pas à ajouter tout information qui peut aider l'utilisateur dans sa demande.
Ajoute aussi des références juridique dans tes reponse(loi,decret,ordonnance ect....).
Ne montre aucun signe de doute de ta part qui pourrait emmener l'utilisateur à douter de tes reponses.
"""

In [86]:
from typing import List

In [87]:
def extract_context_from_documents(documents: List[Document]) -> str:
    """
    Extrait proprement le contenu texte d'une liste de Documents LangChain
    en préservant le format d'affichage.
    """
    extracted_texts = []
    for i, doc in enumerate(documents, start=1):
        # Ajout d'un séparateur pour chaque document (utile pour le prompt)
        
        if "producer" not in doc.metadata.keys():
            section_header = f"\n--- Document {i} ---\n"
            content = doc.page_content.strip()
            provenance = doc.metadata["description"]
            email = doc.metadata["email"]
            telephone = doc.metadata["telephone"]
            source = doc.metadata["source"]
            siege = doc.metadata["siege"]
            extracted_texts.append(section_header + content + email + siege+ telephone+provenance+ "\n"+source)
    return "\n".join(extracted_texts)

In [77]:
retriever = vectorstore.as_retriever()

In [78]:
d = retriever.invoke("C'est quoi un ACD")

In [54]:
from langchain_core.runnables import RunnablePassthrough,RunnableLambda
from operator import itemgetter

In [79]:
from langchain_core.prompts import  ChatPromptTemplate

In [88]:
from langchain_core.output_parsers import StrOutputParser

In [89]:
prompt_template = ChatPromptTemplate.from_template(prompt)

In [90]:
chain ={"question": RunnablePassthrough(),"contexte":retriever|extract_context_from_documents}|prompt_template|model|StrOutputParser()

In [99]:
answer = chain.invoke("quelle loi porte sur l'attestation domaniale")

In [100]:
answer

"# Attestation Domaniale\n\n### Contexte Juridique\n\nL'attestation domaniale est un document important dans le cadre des transactions immobilières en Côte d'Ivoire. Elle formalise la reconnaissance par l'État des droits de propriété d'un individu ou d'une entité sur une parcelle de terrain. Bien que le contexte donné ne fournisse pas de détails précis sur la législation spécifique se rapportant à l'attestation domaniale, il est généralement régi par des lois et décrets nationaux en matière de gestion foncière et domaniale.\n\n### Législation Pertinente\n\nEn Côte d'Ivoire, la gestion foncière est principalement encadrée par les lois suivantes :\n\n1. **Loi n° 98-750 du 23 décembre 1998** : Cette loi régit le domaine foncier rural en Côte d'Ivoire et précise les modalités de reconnaissance des droits fonciers coutumiers.\n   \n2. **Loi n° 2019-868 du 14 octobre 2019** : Relative au Code de la construction et de l'habitat, elle encadre les différentes certifications et documents nécessa

In [37]:
for doc in d:
    print(doc.page_content)

* Description
Obtention d'un ACD avec La lettre d'attribution, acte par lequel l'administration entend signifier à une personne physique ou morale son intention de lui concéder une parcelle de terrain de son domaine privé moyennant le versement d'un prix et l'engagement des procédures de mises en valeur.
* Documents à fournir
- Un Dossier Technique comprenant :
- Un (01) calque ;
- Vingt et une (21) copies photocopies du calque en format A3 ;
- Un (01) tableau de calcul de surfaces ;
- Un (01) rapport du géomètre ;
- Des calculs liés au géo-référencement de la parcelle ;
- Un (01) plan de situation ou de localisation au format A4 ou A3.
- Une (01) fiche de demande d’ACD ;
- Une (01) fiche de renseignement cadastrale ;
- L’original + quatre (04) photocopies de la lettre d’attribution ;
- Quatre (04) photocopies de la pièce d’identité dont une (01) en couleur.
NB : Joindre (04) photocopies des statuts pour les personnes morales.
* Coût
Personnes physiques : 90 000 FCFA/Lot+1 000 FCFA/CHE

In [183]:
retriever_10 = mixed_vector_store.as_retriever()

In [ ]:
vector_store.add_documents(docs_split)

In [ ]:
docs_2 = vector_store.similarity_search("""Je veux acheter un terrain, que dois-je faire?""")


In [ ]:
print(docs_2)

In [ ]:
retriever = vector_store.as_retriever()

In [192]:
document_1 = retriever.invoke("C'est quoi l'attestation domaniale?")

In [221]:
question = """
Tu est un assistant au service du Ministere de la construction et du logement
de Côte d'ivoire, ton rôle sera de repondre aux questions des
utilisateur en te basant sur le conteste suivant:
voici le contexte:
<contexte>
{context}
</contexte>
voici la question:
<question>
{question}
<question>
Renvoie le le resultat au format markdown
n'hésite pas à ajouter tout information qui peut aider l'utilisateur dans sa demande.
Ajoute aussi des références juridique dans tes reponse(loi,decret,ordonnance ect....).
Ne montre aucun signe de doute de ta part qui pourrait emmener l'utilisateur à douter de tes reponses.
"""

In [222]:
question_template = ChatPromptTemplate.from_template(question)

In [223]:
from langchain.schema import Document
from typing import List

def extract_context_from_documents(documents: List[Document]) -> str:
    """
    Extrait proprement le contenu texte d'une liste de Documents LangChain
    en préservant le format d'affichage.
    """
    extracted_texts = []
    for i, doc in enumerate(documents, start=1):
        # Ajout d'un séparateur pour chaque document (utile pour le prompt)

        if "producer" not in doc.metadata.keys():
            section_header = f"\n--- Document {i} ---\n"
            content = doc.page_content.strip()
            provenance = doc.metadata["description"]
            email = doc.metadata["email"]
            telephone = doc.metadata["telephone"]
            source = doc.metadata["source"]
            siege = doc.metadata["siege"]
            extracted_texts.append(section_header + content + email + siege+ telephone+provenance+ "\n"+source)


    return "\n".join(extracted_texts)

In [224]:
from operator import itemgetter

In [225]:
chain = RunnablePassthrough()|{
    "context":retriever|extract_context_from_documents,
    "question":RunnablePassthrough()
}|question_template|model|StrOutputParser()

In [228]:
answer = chain.invoke("""Quelles sont les limites juridiques d’une attestation villageoise ?
Est-elle suffisante pour détenir un terrain?
""")

In [229]:
answer

"\n### Limites Juridiques d'une Attestation Villageoise\n\nL'attestation villageoise est souvent une reconnaissance informelle ou communautaire des droits coutumiers d'une personne ou d'un groupe sur une parcelle de terre. Cependant, elle n'a pas de valeur légale équivalente à un certificat foncier ou à un titre foncier.\n\n#### Est-elle suffisante pour détenir un terrain ?\n\nNon, l'attestation villageoise en elle-même n'est pas suffisante pour détenir légalement un terrain. Pour sécuriser et officialiser les droits fonciers, il est impératif de passer par la procédure de délivrance d'un Certificat Foncier ou d'un Titre Foncier, qui sont des documents légaux reconnus par l'État.\n\n### Processus Légal pour Détention de Terrain\n\n1. **Obtention d'un Certificat Foncier**: \n   - Il est délivré à l'issue d'une enquête officielle supervisée par le Comité Sous-Préfectoral de Gestion Foncière Rurale (CSPGFR), tel que stipulé par le décret n° 2019-266 du 27 mars 2019 et la loi n° 98-750 du 

```markdown
Pour l'achat d'un terrain à Anyama ou ailleurs en Côte d'Ivoire, il est important de vérifier certains aspects légaux pour vous assurer que la transaction est sécurisée et légale. Voici les étapes à suivre :

1. **Vérification de la Lettre d'Attribution** :
   - Assurez-vous que la lettre d'attribution a été émise par l'autorité compétente. En Côte d'Ivoire, les attributions de terrains sont généralement gérées par l'État ou les municipalités. La lettre doit comporter le cachet officiel et la signature de l'autorité compétente.

2. **Authenticité du Document** :
   - Consultez l'administration foncière ou le service des domaines pour vérifier l'authenticité de la lettre d'attribution. Ils peuvent vous indiquer si le document est légitime et si le terrain appartient effectivement au vendeur.

3. **Vérification du Cadastre** :
   - Rendez-vous au bureau cadastral pour obtenir des informations sur le statut du terrain. Vous devez vous assurer qu'il n'est pas déjà vendu à un tiers, grevé de servitudes, ou impliqué dans un litige foncier.

4. **Demande de Titres Fonciers** :
   - Vérifiez si un titre foncier existe déjà pour le terrain. S'il n’y en a pas, demandez au vendeur s'il est possible d'obtenir un document de propriété officiel.

5. **Consultation Juridique** :
   - Il est fortement conseillé de consulter un avocat spécialisé en droit foncier pour vous aider à examiner la légalité du document et de la transaction dans son ensemble.

6. **Autres Documents Relatifs à la Propriété** :
   - Demandez à voir d'autres documents pertinents tels que des actes notariés ou des certificats d'urbanisme, qui peuvent également prouver la légalité de la possession et des transactions précédentes.

En suivant ces étapes, vous pourrez vous assurer que la transaction est en règle et éviter les conflits ou les fraudes dans l'achat de votre terrain.

```

```markdown
Pour l'achat d'un terrain à Anyama ou ailleurs en Côte d'Ivoire, il est important de vérifier certains aspects légaux pour vous assurer que la transaction est sécurisée et légale. Voici les étapes à suivre :

1. **Vérification de la Lettre d'Attribution** :
   - Assurez-vous que la lettre d'attribution a été émise par l'autorité compétente. En Côte d'Ivoire, les attributions de terrains sont généralement gérées par l'État ou les municipalités. La lettre doit comporter le cachet officiel et la signature de l'autorité compétente.

2. **Authenticité du Document** :
   - Consultez l'administration foncière ou le service des domaines pour vérifier l'authenticité de la lettre d'attribution. Ils peuvent vous indiquer si le document est légitime et si le terrain appartient effectivement au vendeur.

3. **Vérification du Cadastre** :
   - Rendez-vous au bureau cadastral pour obtenir des informations sur le statut du terrain. Vous devez vous assurer qu'il n'est pas déjà vendu à un tiers, grevé de servitudes, ou impliqué dans un litige foncier.

4. **Demande de Titres Fonciers** :
   - Vérifiez si un titre foncier existe déjà pour le terrain. S'il n’y en a pas, demandez au vendeur s'il est possible d'obtenir un document de propriété officiel.

5. **Consultation Juridique** :
   - Il est fortement conseillé de consulter un avocat spécialisé en droit foncier pour vous aider à examiner la légalité du document et de la transaction dans son ensemble.

6. **Autres Documents Relatifs à la Propriété** :
   - Demandez à voir d'autres documents pertinents tels que des actes notariés ou des certificats d'urbanisme, qui peuvent également prouver la légalité de la possession et des transactions précédentes.

En suivant ces étapes, vous pourrez vous assurer que la transaction est en règle et éviter les conflits ou les fraudes dans l'achat de votre terrain.

```

In [ ]:
doc = retriever.invoke("Quelle sont les documents à fournir pour la demande d'un ACD à partir de l'attestation")

In [ ]:
doc

In [ ]:
chain.invoke({"question":"Quelle sont les documents à fournir pour la demande d'un ACD à partir de l'attestation"})

In [ ]:
for i in range(len(docs)):
  print(docs[i].page_content)

In [ ]:
for doc in docs:
 print( doc.page_content)

In [ ]:
from langchain_community.document_loaders import UnstructuredHTMLLoader

In [ ]:
%pip install unstructured

In [ ]:
def create_form_html_loader(url: str):
    """
    Crée un UnstructuredHTMLLoader configuré pour extraire spécifiquement le contenu des balises <form>.
    """
    return UnstructuredHTMLLoader(
        file_path=url,
        # IMPORTANT : Le paramètre html_tags est la clé ici.
        # Il indique à Unstructured de ne parser que le contenu à l'intérieur des balises spécifiées.
        html_tags=["form"]
    )

In [ ]:
base_url = "https://construction.gouv.ci/lexique/"

In [ ]:
loader = RecursiveUrlLoader(
    url=base_url,
    loader_cls=create_form_html_loader,
)

In [ ]:
docs = loader.load()

In [ ]:
docs

In [ ]:
print(f"Nombre de documents (formulaires) trouvés : {len(docs)}")
for i, doc in enumerate(docs):
    print(f"\n--- Contenu du formulaire {i+1} ---")
    print(f"Source URL: {doc.metadata.get('source')}")
    print(f"Content: {doc.page_content[:500]}...")

In [ ]:
import json

In [ ]:
d=[{"name": "shakshi","age": 21},{"name": "Kakashi","age": 33}]
print(json.dumps(d))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%pip install langchain

In [ ]:
%pip install pypdf

In [ ]:
%pip install langchain_community

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
file_path = "/content/drive/MyDrive/MASTER1_CORRO/ASD/COURS_ASD_COMPLEXITE (1).pdf"

In [ ]:
loader = PyPDFLoader(file_path)

In [ ]:
pages = loader.load()

In [ ]:
for page in pages:
    print(page.metadata)

In [ ]:
!pip install -qU langchain-community unstructured

In [ ]:
!pip install pdfminer

In [ ]:
from langchain_community.document_loaders import UnstructuredPDFLoader

In [ ]:
!pip install langchain_unstructured

In [ ]:
!pip install pdfminer

In [ ]:
loader = UnstructuredPDFLoader(file_path , mode = "elements")

In [ ]:
%pip install "unstructured[pdf]

In [ ]:
from langchain_unstructured import UnstructuredLoader

loader = UnstructuredLoader(
    file_path=file_path,
    strategy="hi_res"
)
docs = []

In [ ]:
for doc in loader.load():
    docs.append(doc)